## 1. 데이터 준비 및 진행 구조

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

# 입력
x = torch.tensor([
    [1.0],
    [2.0],
    [3.0],
    [4.0]
])

# 정답
y = torch.tensor([
    [2.0],
    [4.0],
    [6.0],
    [8.0]
])

# 선형 모델
model = nn.Linear(1, 1)

# MSE
criterion = nn.MSELoss()

# SGD
optimizer = optim.SGD(
    model.parameters(),
    lr=0.01
)

for epoch in range(1000):

    # 예측
    prediction = model(x)

    # Loss
    loss = criterion(prediction, y)

    # 기존 gradient 삭제
    optimizer.zero_grad()

    # 미분
    loss.backward()

    # 가중치 업데이트
    optimizer.step()

print("weight:", model.weight.item())
print("bias:", model.bias.item())
print("loss:", loss.item())

weight: 1.9855906963348389
bias: 0.042364656925201416
loss: 0.0003015485708601773


5. CNN 학습

이제 실제 사진을 학습시키자.

먼저 필요한 라이브러리:

In [ ]:
pip install torch torchvision opencv-python pillow

Note: you may need to restart the kernel to use updated packages.


In [10]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split


# =========================
# 1. 이미지 전처리
# =========================

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])


# =========================
# 2. 데이터 불러오기
# =========================

dataset = datasets.ImageFolder(
    "../dataset",
    transform=transform
)

print("클래스:", dataset.classes)
print("전체 이미지:", len(dataset))


# =========================
# 3. 학습 / 검증 데이터 분리
# =========================

train_size = int(len(dataset) * 0.8)
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False
)


# =========================
# 4. CNN
# =========================

class CNN(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(64 * 16 * 16, 128),
            nn.ReLU(),

            nn.Linear(128, num_classes)
        )

    def forward(self, x):

        x = self.features(x)
        x = self.classifier(x)

        return x


# =========================
# 5. 모델 생성
# =========================

num_classes = len(dataset.classes)

model = CNN(num_classes)

print(model)


# =========================
# 6. Loss / Optimizer
# =========================

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)


# =========================
# 7. 학습
# =========================

epochs = 20

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for images, labels in train_loader:

        prediction = model(images)

        loss = criterion(
            prediction,
            labels
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch + 1}/{epochs} "
        f"Loss: {total_loss:.4f}"
    )


# =========================
# 8. 모델 저장
# =========================

torch.save(
    {
        "model_state": model.state_dict(),
        "classes": dataset.classes
    },
    "models/object_cnn.pth"
)

print("학습 완료!")
print("모델 저장 완료!")

클래스: ['nipper', 'scraper']
전체 이미지: 59
CNN(
  (features): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=16384, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=2, bias=True)
  )
)
Epoch 1/20 Loss: 8.4524
Epoch 2/20 Loss: 8.3119
Epoch 3/20 Loss: 5.9552
Epoch 4/20 Loss: 0.9391
Epoch 5/20 Loss: 0.4885
Epoch 6/20 Loss: 0.0217
Epoch 7/20 Loss: 0.0186
Epoch 8/20 Loss: 0.

In [9]:
import os
import torch

# models 폴더 자동 생성
os.makedirs("models", exist_ok=True)

# 모델 저장
torch.save(
    {
        "model_state": model.state_dict(),
        "classes": dataset.classes
    },
    "models/object_cnn.pth"
)

print("🎉 학습 완료!")
print("💾 모델 저장 완료!")
print("클래스:", dataset.classes)

🎉 학습 완료!
💾 모델 저장 완료!
클래스: ['nipper', 'scraper']


이 코드에서:

nn.Conv2d(3, 16, 3, padding=1)

의 3은 RGB 이미지의 채널 수야.

그리고:

nn.MaxPool2d(2)

를 거치면서 이미지 크기를 줄이고 특징을 압축해.

마지막:

nn.Linear(128, num_classes)

에서 최종적으로

스크래퍼
렌치

중 어느 것인지 판단한다.

In [11]:
nn.Conv2d(3, 16, 3, padding=1)

Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))